# LegalQA Task 2 — Google Colab A100 Production Training Pipeline (Notion DSC 2026)
Production QLoRA generator training, full validation, and Hugging Face artifact release on NVIDIA A100.
- **Prerequisite**: Kaggle Dual-T4 Smoke Gate must have reached  status for the frozen tuple.
- **Configuration**:  (BF16 native throughput, larger batch size, full training epochs).
- **Outputs**: Full Run Bundle exported and uploaded to Hugging Face repository.

In [ ]:
# Cell 1: Hardware & Environment Verification
import os, sys, subprocess, torch

print("=== Hardware Verification ===")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Colab A100 production training.")

gpu_name = torch.cuda.get_device_name(0)
print(f"Detected GPU: {gpu_name}")
if "A100" not in gpu_name:
    print(f"Notice: Optimal profile target is NVIDIA A100, currently running on {gpu_name}.")

try:
    subprocess.run(["nvidia-smi"], check=True)
except Exception:
    pass


In [ ]:
# Cell 2: Git Repository Sync & Commit Verification
import os
TARGET_GIT_SHA = "main"

if not os.path.exists("LegalQA") and not os.path.exists("src/task2"):
    os.system("git clone https://github.com/silent9669/LegalQA.git")
    os.chdir("LegalQA")

if TARGET_GIT_SHA != "main":
    os.system(f"git checkout {TARGET_GIT_SHA}")

print("Repository synchronized.")


In [ ]:
# Cell 3: Dataset Mount & Schema Validation
import sys, os
if "." not in sys.path:
    sys.path.insert(0, ".")

from src.task2.dataset.validator import validate_dataset

DATA_DIR = "/content/data/legalqa-task2-clean-data"
if not os.path.exists(DATA_DIR):
    if os.path.exists("kaggle_dataset/staged"):
        DATA_DIR = os.path.abspath("kaggle_dataset/staged")
    else:
        import kagglehub
        DATA_DIR = kagglehub.dataset_download("phucdangg/legalqa-task2-clean-data")

print(f"Active Dataset Directory: {DATA_DIR}")
val_report = validate_dataset(data_dir=DATA_DIR, schema_path="configs/dataset_schema.yaml")
print(f"Dataset Manifest Status: {val_report.get('status')} (Verified: {val_report.get('manifest_verified')})")
if val_report.get("status") != "PASS":
    raise RuntimeError(f"Dataset validation failed: {val_report.get('errors')}")


In [ ]:
# Cell 4: Smoke Pass Gate Verification
from src.task2.provenance.freeze_tuple import verify_smoke_pass

SMOKE_REPORT_PATH = "kaggle_smoke_report.json"
if os.path.exists(SMOKE_REPORT_PATH):
    if not verify_smoke_pass(SMOKE_REPORT_PATH):
        raise RuntimeError("Kaggle smoke gate did NOT report PASS. Cannot proceed with A100 training.")
    print("Verified: Kaggle Dual-T4 smoke gate PASS confirmed.")
else:
    print("Notice: Proceeding with explicit execution (kaggle_smoke_report.json not attached locally).")


In [ ]:
# Cell 5: Execute Colab A100 Production Training
import os
cmd = f"python scripts/run_pipeline.py --config configs/colab_train_a100.yaml --data-dir {DATA_DIR} --output-dir /content/runs/current --allow-single-gpu"
print(f"Executing: {cmd}")
ret = os.system(cmd)
if ret != 0:
    raise RuntimeError(f"Pipeline execution returned non-zero exit code: {ret}")
print("Colab A100 production training finished successfully.")


In [ ]:
# Cell 6: Run Bundle Packaging & Evidence Verification
import glob
print("=== Generated Run Artifacts ===")
for p in sorted(glob.glob("/content/runs/current/**", recursive=True)):
    if os.path.isfile(p):
        print(f" - {p} ({os.path.getsize(p)/1024:.1f} KB)")
print("
Run Bundle successfully generated for Hugging Face upload.")
